In [5]:
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
import re
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever


from langchain_core.prompts import ChatPromptTemplate

In [ ]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to a .env file.")

In [7]:
loader = TextLoader(r"C:\Resume_Chatbot\dhruv_desai_knowledge_base.txt")

documents = loader.load()

print(f"Loaded {len(documents)} document(s)")

Loaded 1 document(s)


In [8]:
with open("C:\\Resume_Chatbot\\dhruv_desai_knowledge_base.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [9]:
sections = re.split(
    r'(?=SECTION: )',
    text
)

sections = [
    section.strip()
    for section in sections
    if section.strip()
]

In [10]:
print(len(sections))

for section in sections:
    print(section[:100])
    print("-----")

17
DHRUV DESAI — PORTFOLIO KNOWLEDGE BASE
(Source: https://portfolio-website-dhruvdesai.vercel.app/ — e
-----
SECTION: WHO IS DHRUV DESAI (IDENTITY & SUMMARY)
-----
SECTION: CONTACT INFORMATION & AVAILABILITY
-----
SECTION: WORK EXPERIENCE — CURRENT ROLE
-----
SECTION: WORK EXPERIENCE — PREVIOUS ROLE
-----
SECTION: SKILLS — FULL BREAKDOWN BY CATEGORY
-----
SECTION: PROJECT 1 — TRAVEL PLANNING AGENTIC CHATBOT
-----
SECTION: PROJECT 2 — MEDICAL COST PREDICTION
-----
SECTION: PROJECT 3 — CUSTOMER PURCHASE PREDICTION APP
-----
SECTION: PROJECT 4 — CREDIT CARD DEFAULT PREDICTION
-----
SECTION: PROJECT 5 — GLASSDOOR DATA SCIENCE JOBS ANALYSIS
-----
SECTION: PROJECT 6 — TELEGRAM TECHNICAL ANALYST CHATBOT
-----
SECTION: PROJECT 7 — BLINKIT SALES POWER BI DASHBOARD
-----
SECTION: EDUCATION BACKGROUND
-----
SECTION: CERTIFICATIONS
-----
SECTION: PUBLICATION — "AI-BASED ESG AUDITING SYSTEM – 'AUDITOR'"
-----
SECTION: FREQUENTLY ASKED QUESTIONS (FOR HIRERS / RECRUITERS)
-----


In [11]:
documents = [
    Document(
        page_content=section,
        metadata={
            "section": section.split("\n")[0]
        }
    )
    for section in sections
]

In [12]:
documents

[Document(metadata={'section': 'DHRUV DESAI — PORTFOLIO KNOWLEDGE BASE'}, page_content='DHRUV DESAI — PORTFOLIO KNOWLEDGE BASE\n(Source: https://portfolio-website-dhruvdesai.vercel.app/ — extracted for a RAG chatbot that answers hirer/recruiter questions about Dhruv Desai)\n\nThis document is organized into self-contained sections. Each section can be retrieved independently and fully answers questions about that topic. Dhruv Desai is referred to by full name or "Dhruv" throughout so that isolated chunks remain understandable out of context.\n\n================================================================================'),
 Document(metadata={'section': 'SECTION: WHO IS DHRUV DESAI (IDENTITY & SUMMARY)'}, page_content='SECTION: WHO IS DHRUV DESAI (IDENTITY & SUMMARY)\n================================================================================\nDhruv Desai is an Data Scientist and GenAI Developer based in Dubai, UAE (GMT+4). His professional title is "Data Scientist & Gen AI De

In [13]:
embeddings = OpenAIEmbeddings(
    api_key=openai_api_key,
    model="text-embedding-3-small"
)

In [14]:
vector_store = FAISS.from_documents(
    documents,
    embeddings
)

In [15]:
vector_retriever = vector_store.as_retriever(
    search_kwargs={"k": 10}
)

bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 10

# Hybrid search: BM25 (keyword/sparse) + FAISS (semantic/dense)
retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

In [16]:
llm = ChatOpenAI(
    api_key=openai_api_key,
    model="gpt-4o-mini",
    temperature=0
)

In [17]:
prompt = ChatPromptTemplate.from_template(
    """
You are a precise, evidence-grounded question-answering assistant.

Your task is to answer the user's question using ONLY the information explicitly
contained in the provided context.

Follow these rules strictly:

1. CONTEXT IS THE ONLY SOURCE OF TRUTH
   - Do not use outside knowledge.
   - Do not assume, infer, or invent information that is not explicitly supported
     by the context.

2. ANSWER THE EXACT QUESTION
   - Identify what the user is asking before answering.
   - Do not add unrelated information.
   - If the question asks for multiple things, make sure you address EVERY part.

3. MAXIMIZE FACTUAL COMPLETENESS
   - Include all important facts from the context that are directly relevant
     to the question.
   - Do not unnecessarily omit names, dates, organizations, technologies,
     credentials, numbers, skills, or other relevant details.

4. PRESERVE FACTUAL DETAILS
   - Do not change names, numbers, dates, percentages, titles, organizations,
     credential IDs, or technical terms.
   - Do not paraphrase a fact in a way that changes its meaning.

5. HANDLE MISSING INFORMATION HONESTLY
   - If the context does not contain enough information to answer the question,
     explicitly say:
     "I don't have enough information in the provided context to answer that."
   - Never fill missing information using assumptions.

6. DISTINGUISH FACT FROM INFERENCE
   - Only state something as a fact when it is directly supported by the context.
   - If something cannot be confirmed from the context, do not present it as true.

7. FOR LIST/COUNT QUESTIONS
   - Carefully check the entire context before answering.
   - Include all matching items rather than returning only the first few.
   - Do not claim a count unless it can be verified from the context.

8. FOR COMPARISON QUESTIONS
   - Compare only attributes explicitly present in the context.
   - Do not introduce external knowledge to make the comparison.

9. KEEP THE ANSWER CONCISE
   - Give a clear and direct answer.
   - Use bullet points when listing multiple facts.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [18]:
def ask_rag(question):

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    response = llm.invoke(messages)

    return response.content

In [19]:
question = "Tell me about his Certifications."

answer = ask_rag(question)

In [20]:
print(answer)

Dhruv Desai has the following certifications:

1. **Anthropic**
   - Claude Code 101 — Verified certificate
   - Introduction to Model Context Protocol — Verified certificate
   - Introduction to Claude Cowork — Verified certificate

2. **Udemy**
   - Complete Generative AI Course with LangChain and Hugging Face
     - Skills: Generative AI, LangChain, Hugging Face, Natural Language Processing (NLP), Python
     - Credential ID: UC-0de95a68-72a6-4fce-8e3b-162aea85484b

3. **Google**
   - Foundations of Data Science — Issued July 2024
     - Credential ID: GAJT9FTL6Y82
   - Get Started with Python — Issued July 2024
     - Credential ID: UMMBLWGQK452
     - Skill: Python
   - Go Beyond the Numbers: Translate Data into Insights — Issued September 2024
     - Credential ID: AVV4CPX8GG4W
     - Skills: Pandas, Exploratory Data Analysis (EDA)
   - Regression Analysis: Simplify Complex Data Relationships — Issued October 2024
     - Credential ID: UHSAXNB73PSN
     - Skills: Machine Learning

In [21]:
sample_queries = [
    # --- IDENTITY ---
    "Who is Dhruv Desai?",
    "What does Dhruv specialize in?",
    "What is Dhruv's professional philosophy or approach to AI work?",
    "Where is Dhruv based?",
 
    # --- CONTACT ---
    "How can I contact Dhruv?",
    "What is Dhruv's GitHub profile link?",
    "Is Dhruv open to freelance work or collaborations?",
 
    # --- CURRENT EXPERIENCE ---
    "What is Dhruv's current job?",
    "When did Dhruv start his current role, and who is the client?",
    "What technologies does Dhruv use in his current role?",
 
    # --- PREVIOUS EXPERIENCE ---
    "What did Dhruv do at VT Index?",
    "When did Dhruv work at VT Index?",
    "What tech stack did Dhruv use during his internship at VT Index?",
 
    # --- SKILLS ---
    "What programming languages does Dhruv know?",
    "What machine learning frameworks has Dhruv used?",
    "Does Dhruv have experience with agentic AI frameworks?",
    "What MLOps tools does Dhruv know?",
    "Does Dhruv have BI or dashboarding experience?",
    "Does Dhruv have NLP experience?",
 
    # --- EDUCATION ---
    "What is Dhruv's educational background?",
    "Did Dhruv graduate with honors or distinction?",
    "What subjects did Dhruv study in Grade 12 / higher secondary school, and what was his score?",
    "What topics did Dhruv's Bachelor's degree cover?",
 
    # --- CERTIFICATIONS ---
    "What certifications does Dhruv hold?",
    "Does Dhruv have any Google certifications?",
    "Does Dhruv have any certifications related to Claude or Anthropic?",
    "Does Dhruv have any Generative AI or LangChain certifications?",
    "Does Dhruv have a Hugging Face certification?",
 
    # --- PUBLICATION ---
    "Has Dhruv published any research papers?",
    "What is Dhruv's published paper about?",
    "What machine learning or NLP technologies were discussed in Dhruv's ESG auditing paper?",
 
    # --- PROJECT: Travel Planning Agentic Chatbot ---
    "What is the Travel Planning Agentic Chatbot project about?",
    "What technology stack was used in the travel planning chatbot project?",
    "What problem does the travel planning chatbot solve?",
 
    # --- PROJECT: Medical Cost Prediction ---
    "What accuracy did Dhruv achieve on the Medical Cost Prediction project?",
    "What algorithms did Dhruv test for the medical cost prediction project?",
    "What was the strongest predictor of insurance costs in Dhruv's medical cost project?",
 
    # --- PROJECT: Customer Purchase Prediction ---
    "What accuracy did the customer purchase prediction model achieve?",
    "What model performed best for customer purchase prediction, and what were its precision and recall?",
 
    # --- PROJECT: Credit Card Default Prediction ---
    "What accuracy did Dhruv's credit card default prediction model achieve?",
    "What challenges did Dhruv face in the credit card default prediction project?",
 
    # --- PROJECT: Glassdoor Jobs Analysis ---
    "What did Dhruv find in his Glassdoor Data Science jobs analysis?",
    "How large was the dataset in the Glassdoor jobs analysis project?",
 
    # --- PROJECT: Telegram Technical Analyst Chatbot ---
    "How many users does Dhruv's Telegram chatbot serve?",
    "What engagement improvement resulted from Dhruv's Telegram chatbot?",
 
    # --- PROJECT: Blinkit Power BI Dashboard ---
    "What tools did Dhruv use for the Blinkit Power BI dashboard?",
    "What were the key findings from the Blinkit sales dashboard?",
 
    # --- AVAILABILITY ---
    "Is Dhruv available for full-time roles?",
    "How quickly does Dhruv typically respond to inquiries?",
 
    # --- SYNTHESIS (requires combining multiple chunks) ---
    "Has Dhruv built both regression and classification models?",
    "What industries or domains has Dhruv applied his data science work to?",
    "Considering his degree, certifications, and published paper together, how has Dhruv formally validated his AI/ML skills?",
 
    # --- NEGATIVE / OUT-OF-SCOPE (info NOT in the source document) ---
    "What is Dhruv's phone number?",
    "What is Dhruv's expected salary?",
    "What's the weather like in Dubai today?",
    "What exact GPA or percentage did Dhruv score in his Bachelor's degree?",
    "Which board (e.g., CBSE, ICSE) or university was Dhruv's Higher Secondary school affiliated with?",
]
 
expected_responses = [
    # --- IDENTITY ---
    "Dhruv Desai is an AI & Data Science Engineer based in Dubai, UAE, specializing in machine learning, agentic AI systems, MLOps, and automation. He currently works as an RTIM Business Architect at Elitser Technology, deputed to Emirates NBD (ENBD).",
    "Dhruv specializes in data science, generative AI (autonomous agent pipelines and LLM orchestration), n8n workflow automation, and end-to-end data analysis with SQL-driven insights.",
    "Dhruv's stated philosophy is building 'scalable AI systems that ship to production — not just notebook experiments,' reflecting his focus on deploying real, working systems rather than only research prototypes.",
    "Dhruv is based in Dubai, UAE (GMT+4).",
 
    # --- CONTACT ---
    "Dhruv can be reached by email at dkdesai2004@gmail.com or via LinkedIn at https://linkedin.com/in/dhruv-desai-b35b171b6/.",
    "Dhruv's GitHub profile is https://github.com/dhruvatgithub2004.",
    "Yes. Dhruv's portfolio states he is open to full-time, freelance, and collaboration opportunities, available remotely or on-site, and typically responds within 24 hours.",
 
    # --- CURRENT EXPERIENCE ---
    "Dhruv's current job is RTIM Business Architect at Elitser Technology, deputed to Emirates NBD (ENBD), focused on data analysis, real-time decisioning, and workflow automation.",
    "Dhruv started his current role in April 2026, working for Elitser Technology while deputed to the client Emirates NBD (ENBD).",
    "In his current role, Dhruv uses Python, n8n, SQL, data pipelines, and workflow automation.",
 
    # --- PREVIOUS EXPERIENCE ---
    "At VT Index, Dhruv worked as an AI & Automation Intern and built a Telegram trading bot that served 500+ active users, automating technical market analysis that previously required manual research.",
    "Dhruv worked at VT Index from December 2025 to January 2026.",
    "During his internship at VT Index, Dhruv used Python, n8n, the Telegram API, and the Google Sheets API.",
 
    # --- SKILLS ---
    "Python is Dhruv's core programming language, used across nearly all of his projects.",
    "Dhruv has used Scikit-learn and TensorFlow as his primary machine learning frameworks.",
    "Yes. Dhruv has experience with agentic AI frameworks including Google ADK, n8n, LangGraph, and LangChain — notably building a multi-agent travel chatbot with Google ADK.",
    "Dhruv's MLOps and infrastructure skills include Docker, MLflow, FastAPI, Kubernetes, and Streamlit.",
    "Yes. Dhruv built an interactive Power BI dashboard (the Blinkit Sales dashboard) using Power BI, SQL, and DAX.",
    "NLP is listed among Dhruv's AI & machine learning skills, alongside Scikit-learn, TensorFlow, and GRU/RNN.",
 
    # --- EDUCATION ---
    "Dhruv holds a Bachelor of Computer Applications (BCA) specializing in Artificial Intelligence (2022–2025), graduating with Distinction, and completed his Higher Secondary Education at an Indian High School with 85%, specializing in Economics, Business Studies, Accountancy, and Mathematics.",
    "Yes. Dhruv graduated with Distinction in his three-year BCA (Artificial Intelligence) program.",
    "In Grade 12, Dhruv specialized in Economics, Business Studies, Accountancy, and Mathematics, and completed his Higher Secondary Education with a score of 85%.",
    "Dhruv's BCA (Artificial Intelligence) program covered Statistics, Machine Learning, Python, Pandas, Data Visualization, Relational Database Management Systems (RDBMS), and Deep Learning.",
 
    # --- CERTIFICATIONS ---
    "Dhruv holds certifications including three verified Anthropic certificates (Claude Code 101, Introduction to Model Context Protocol, and Introduction to Claude Cowork), Udemy's 'Complete Generative AI Course with LangChain and Hugging Face,' five Google certifications (Foundations of Data Science; Get Started with Python; Go Beyond the Numbers: Translate Data into Insights; Regression Analysis: Simplify Complex Data Relationships; The Power of Statistics), and Hugging Face's 'AI Agents Fundamentals.'",
    "Yes. Dhruv holds five Google certifications: Foundations of Data Science (issued July 2024), Get Started with Python (issued July 2024), Go Beyond the Numbers: Translate Data into Insights (issued September 2024), Regression Analysis: Simplify Complex Data Relationships (issued October 2024), and The Power of Statistics (issued October 2024).",
    "Yes. Dhruv holds three verified Anthropic certificates: Claude Code 101, Introduction to Model Context Protocol (MCP), and Introduction to Claude Cowork.",
    "Yes. Dhruv completed Udemy's 'Complete Generative AI Course with LangChain and Hugging Face,' covering Generative AI, LangChain, Hugging Face, NLP, and Python.",
    "Yes. Dhruv holds a 'Hugging Face AI Agents Fundamentals' certification.",
 
    # --- PUBLICATION ---
    "Yes. Dhruv is a co-author (with Abir Rawat and Riya Matekar) of 'AI-Based ESG Auditing System – \"AudItor\"', published in the International Journal of Advanced Research in Science, Engineering and Technology (IJARSET), Vol. 12, Issue 3, March 2025.",
    "The paper proposes 'AudItor,' an AI-powered system that automates ESG (Environmental, Social, and Governance) auditing by estimating a company's ESG score from structured and unstructured data and providing tailored sustainability recommendations through an integrated AI advisory chatbot.",
    "The paper discusses Random Forest and XGBoost for ESG score prediction, LSTM for ESG trend forecasting, Doc2Vec for converting ESG reports into vector representations, Transformer-based models for extracting context-aware insights from ESG disclosures, ESG-BERT for financial/sustainability text classification, and T5 for turning unstructured ESG text into structured insights.",
 
    # --- PROJECT: Travel Planning Agentic Chatbot ---
    "The Travel Planning Agentic Chatbot is a multi-agent travel concierge built with Google ADK that answers natural-language destination, place, and travel-news queries using live web search and map data, replacing the need to juggle separate search engines, review sites, and news sources.",
    "The travel planning chatbot was built with Python, Google ADK (agent framework), LiteLLM (model interface), FastAPI (backend), Streamlit (frontend), the Tavily API (search), and OpenStreetMap/Overpass/Nominatim (geospatial data), using the DeepSeek Chat model.",
    "It solves the problem of trip planning requiring disconnected sources (search engines, maps/review sites, news sites) by consolidating destination, place, and travel-news queries into a single conversational interface with real-time, grounded information.",
 
    # --- PROJECT: Medical Cost Prediction ---
    "Dhruv's Medical Cost Prediction project achieved a 99% R² score on the test set.",
    "Dhruv tested Linear Regression, Ridge, Lasso, and Random Forest Regressor, tuned with GridSearchCV and validated via cross-validation.",
    "Smoking status was the strongest predictor of insurance charges in that project, with BMI (especially above 30) and age also showing positive relationships with cost.",
 
    # --- PROJECT: Customer Purchase Prediction ---
    "The best model (Random Forest) achieved 93% accuracy on the customer purchase prediction project.",
    "Random Forest was the best-performing model, achieving 93% accuracy, 94% precision, and 90% recall.",
 
    # --- PROJECT: Credit Card Default Prediction ---
    "The best-generalizing model, Logistic Regression, achieved 72.5% accuracy (with a 0.18 F1-score) on the credit card default prediction project.",
    "Key challenges included severe overfitting on the Random Forest model (100% train accuracy vs. ~70% test), class imbalance, high dimensionality (87 features), and weak F1-scores on the minority (default) class.",
 
    # --- PROJECT: Glassdoor Jobs Analysis ---
    "Dhruv found that Python and SQL appear in over 80% of Data Science job postings, senior roles pay 2-3x entry-level compensation, tech and finance sectors offer the highest average salaries, and cloud skills are increasingly required at mid-to-senior levels.",
    "The Glassdoor jobs analysis used a dataset of 120,000+ job posting records.",
 
    # --- PROJECT: Telegram Technical Analyst Chatbot ---
    "Dhruv's Telegram Technical Analyst Chatbot serves 500+ active users.",
    "The Telegram chatbot delivered a 40% engagement uplift compared to the prior manual research workflow.",
 
    # --- PROJECT: Blinkit Power BI Dashboard ---
    "Dhruv used Power BI Desktop, SQL, Python, and DAX for the Blinkit Sales Power BI dashboard.",
    "Key findings were that Fruits & Vegetables and Snack Foods drive the highest revenue, Tier 3 outlets outperform Tier 1 in sales volume, regular fat-content products outsell low-fat alternatives, and outlets established in 2018 or earlier show higher per-item sales.",
 
    # --- AVAILABILITY ---
    "Yes, Dhruv's portfolio states he is open to full-time positions, in addition to freelance work and collaborations.",
    "Dhruv typically responds to inquiries within 24 hours.",
 
    # --- SYNTHESIS ---
    "Yes. Dhruv has built regression models (e.g., Medical Cost Prediction, 99% R²) and classification models (e.g., Customer Purchase Prediction at 93% accuracy, Credit Card Default Prediction at 72.5% accuracy).",
    "Dhruv's projects span insurance (medical cost prediction), e-commerce/retail (customer purchase prediction, Blinkit sales dashboard), banking/finance (credit card default prediction, and his current ENBD-related role), travel (the agentic travel chatbot), trading/financial markets (the Telegram technical analysis bot), and recruitment/hiring analytics (the Glassdoor jobs analysis).",
    "Dhruv has validated his AI/ML skills formally through a Bachelor's degree (BCA, Artificial Intelligence, 2022–2025, graduated with Distinction), multiple certifications (Anthropic, Google, Udemy, Hugging Face) covering Python, statistics, regression, generative AI, and LangChain, and a peer-reviewed publication (the IJARSET 'AudItor' ESG auditing paper) — in addition to his hands-on portfolio projects.",
 
    # --- NEGATIVE / OUT-OF-SCOPE ---
    "This information is not available — no phone number is listed for Dhruv.",
    "This information is not available — Dhruv's portfolio does not list salary expectations.",
    "This is outside the scope of what I know about Dhruv Desai — I can only answer questions about his background, skills, experience, and projects.",
    "This information is not available — the source only states that Dhruv graduated with Distinction; no specific GPA or percentage is given for his Bachelor's degree.",
    "This information is not available — the source only names 'Indian High School' for Dhruv's Higher Secondary Education; it does not specify a board or affiliated university.",
]

In [22]:
from ragas import EvaluationDataset

dataset = []

for query, reference in zip(sample_queries, expected_responses):

    # 1. Retrieve relevant documents
    relevant_docs = retriever.invoke(query)

    # 2. Format retrieved documents into context
    context = "\n\n".join(
        doc.page_content for doc in relevant_docs
    )

    # 3. Create prompt using your existing prompt
    messages = prompt.invoke({
        "context": context,
        "question": query
    })

    # 4. Generate answer using your OpenAI LLM
    response = llm.invoke(messages)

    # 5. Store everything needed by Ragas
    dataset.append({
        "user_input": query,
        "retrieved_contexts": [
            doc.page_content for doc in relevant_docs
        ],
        "response": response.content,
        "reference": reference
    })

# Create Ragas evaluation dataset
evaluation_dataset = EvaluationDataset.from_list(dataset)


c:\Resume_Chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from datasets import Dataset

evaluator_llm = LangchainLLMWrapper(llm)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[context_recall, faithfulness, context_precision, answer_relevancy],
    llm=evaluator_llm,
    embeddings=embeddings,
)

result

C:\Users\dkdes\AppData\Local\Temp\ipykernel_11676\1993339463.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\dkdes\AppData\Local\Temp\ipykernel_11676\1993339463.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\dkdes\AppData\Local\Temp\ipykernel_11676\1993339463.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\dkdes\AppData\Local\Temp\ipykernel_11

{'context_recall': 0.9650, 'faithfulness': 0.9459, 'context_precision': 0.7294, 'answer_relevancy': 0.7900}